In [1]:
!nvidia-smi

Mon Nov  9 04:32:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q transformers datasets peft accelerate jiwer soundfile librosa pydantic

In [3]:
import torch
from transformers import WhisperForConditionalGeneration, WhisperProcessor

MODEL_ID = "distil-whisper/distil-small.en"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

processor = WhisperProcessor.from_pretrained(MODEL_ID)
model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device).eval()

print("Model loaded. Params:", round(sum(p.numel() for p in model.parameters()) / 1e6, 1), "M")

Device: cuda


preprocessor_config.json:   0%|          | 0.00/339 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.26k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/282k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/999k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.41M [00:00<?, ?B/s]

... (5 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/287 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/1.92k [00:00<?, ?B/s]

Model loaded. Params: 166.1 M


In [4]:
from datasets import load_dataset

ds = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean", split="validation")
audio = ds[0]["audio"]["array"]
print("Audio length (sec):", round(len(audio) / 16000, 1))
print("Reference text:", ds[0]["text"])

feats = processor(audio, sampling_rate=16000, return_tensors="pt").input_features
feats = feats.to(device, dtype=model.dtype)

with torch.no_grad():
    ids = model.generate(feats, max_new_tokens=225)

print("Model output:", processor.batch_decode(ids, skip_special_tokens=True)[0])

README.md:   0%|          | 0.00/520 [00:00<?, ?B/s]

clean/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 9.19MB            

clean/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating validation split:   0%|          | 0/73 [00:00<?, ? examples/s]

[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Audio length (sec): 5.9
Reference text: MISTER QUILTER IS THE APOSTLE OF THE MIDDLE CLASSES AND WE ARE GLAD TO WELCOME HIS GOSPEL


[transformers] Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Model output:  Mr. Quilter is the Apostle of the Middle Classes, and we are glad to welcome his Gospel.


In [5]:
import numpy as np

with torch.no_grad():
    out = model.generate(feats, return_dict_in_generate=True,
                         output_scores=True, max_new_tokens=225)

logprobs = model.compute_transition_scores(
    out.sequences, out.scores, normalize_logits=True
)[0].float().cpu().numpy()

gen_ids = out.sequences[0, -len(out.scores):].tolist()

print(f"{'TOKEN':<15} CONFIDENCE")
for tid, lp in zip(gen_ids, logprobs):
    print(f"{processor.tokenizer.decode([tid])!r:<15} {np.exp(lp):.3f}")

[transformers] The following generation flags are not valid and may be ignored: ['output_scores']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


TOKEN           CONFIDENCE
' Mr'           0.959
'.'             0.939
' Qu'           0.959
'il'            0.862
'ter'           0.997
' is'           0.990
' the'          0.995
' Apostle'      0.849
' of'           0.996
' the'          0.970
' Middle'       0.953
' Classes'      0.483
','             0.640
' and'          0.991
' we'           0.998
' are'          0.945
' glad'         0.994
' to'           0.998
' welcome'      0.996
' his'          0.883
' Gospel'       0.565
'.'             0.969
'<|endoftext|>' 0.987


In [6]:
import string

special = set(processor.tokenizer.all_special_ids)

def merge_words(gen_ids, logprobs):
    words, cur = [], None
    for tid, lp in zip(gen_ids, logprobs):
        if tid in special:
            continue
        piece = processor.tokenizer.decode([tid])
        stripped = piece.strip()
        is_punct = stripped != "" and all(c in string.punctuation for c in stripped)

        if piece.startswith(" ") or cur is None:      # new word starts
            if cur:
                words.append(cur)
            cur = {"text": stripped, "lps": [float(lp)]}
        else:                                          # continuation of the same word
            cur["text"] += piece
            if not is_punct:                           # punctuation doesn't affect confidence
                cur["lps"].append(float(lp))
    if cur:
        words.append(cur)

    for w in words:
        probs = np.exp(w["lps"])
        w["conf_geo"] = float(np.exp(np.mean(w["lps"])))   # geometric mean
        w["conf_min"] = float(np.min(probs))               # most conservative
    return words

words = merge_words(gen_ids, logprobs)

TAU = 0.90
print(f"{'WORD':<14}{'GEO':>7}{'MIN':>7}  FLAG")
for w in words:
    flag = "LOW" if w["conf_geo"] < TAU else ""
    print(f"{w['text']:<14}{w['conf_geo']:>7.3f}{w['conf_min']:>7.3f}  {flag}")

WORD              GEO    MIN  FLAG
Mr.             0.959  0.959  
Quilter         0.938  0.862  
is              0.990  0.990  
the             0.995  0.995  
Apostle         0.849  0.849  LOW
of              0.996  0.996  
the             0.970  0.970  
Middle          0.953  0.953  
Classes,        0.483  0.483  LOW
and             0.991  0.991  
we              0.998  0.998  
are             0.945  0.945  
glad            0.994  0.994  
to              0.998  0.998  
welcome         0.996  0.996  
his             0.883  0.883  LOW
Gospel.         0.565  0.565  LOW


In [7]:
import time
import string

def clean_word(w: str) -> str:
    return w.strip(string.punctuation + " ")

def transcribe_with_confidence(audio, sr=16000, mode="geo"):
    t0 = time.time()
    feats = processor(audio, sampling_rate=sr, return_tensors="pt").input_features
    feats = feats.to(device, dtype=model.dtype)

    with torch.no_grad():
        out = model.generate(feats, return_dict_in_generate=True,
                             output_scores=True, max_new_tokens=225)
    lps = model.compute_transition_scores(
        out.sequences, out.scores, normalize_logits=True)[0].float().cpu().numpy()
    ids = out.sequences[0, -len(out.scores):].tolist()

    words = merge_words(ids, lps)
    key = "conf_geo" if mode == "geo" else "conf_min"
    return {
        "raw_transcript": processor.batch_decode(out.sequences, skip_special_tokens=True)[0].strip(),
        "word_confidences": [(clean_word(w["text"]), round(w[key], 4))
                             for w in words if clean_word(w["text"])],
        "utterance_confidence": round(float(np.exp(np.mean(
            [lp for tid, lp in zip(ids, lps) if tid not in special]))), 4),
        "latency_ms": round((time.time() - t0) * 1000, 1),
    }

res = transcribe_with_confidence(audio)
print(res["raw_transcript"])
print("Utterance conf:", res["utterance_confidence"], "| Latency:", res["latency_ms"], "ms")
print("LOW words:", [w for w in res["word_confidences"] if w[1] < 0.90])

[transformers] Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Mr. Quilter is the Apostle of the Middle Classes, and we are glad to welcome his Gospel.
Utterance conf: 0.8909 | Latency: 327.9 ms
LOW words: [('Apostle', 0.8491), ('Classes', 0.4826), ('his', 0.8829), ('Gospel', 0.5652)]
